# Remote CDS

Data downloaded from the Copernicus Climate Data Store, for the points only, and kept in a cache.

Before the first use:
1. create an account and write the key in `~/.cdsapirc`: https://cds.climate.copernicus.eu/how-to-api
2. accept the licence of the dataset on its page (ERA5 single levels), or CDS refuses the requests

In [1]:
import pandas as pd
import cdsapi

from floatmatcher.floatmatcher import FloatMatcher
from floatmatcher.mocking import FakeCDSClient

client = cdsapi.Client()

2026-11-07 00:38:35,110 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.


### Points and product

In [2]:
df = pd.DataFrame({
    "longitude": [-38.2, -35.6, -31.9, -27.4, -23.1],
    "latitude":  [31.5, 34.2, 37.8, 40.1, 43.6],
    "date": pd.to_datetime(["2018-01-02 03:10", "2018-01-03 14:45", "2018-01-05 08:20",
                            "2018-01-06 21:05", "2018-01-08 11:30"]),
})

my_fm = FloatMatcher()
my_fm.set_points(df)
my_fm.set_product(type="era5-hourly",
                  source="remote-cds",
                  path="./cache",
                  selected_variables=["sst"],
                  client=client,
                  time=["00:00", "06:00", "12:00", "18:00"],  # fewer hours, smaller request
                  )

### Request, before downloading

In [3]:
my_fm.build_request()  # area of the points plus a margin, their days, the hours asked

[{'product_type': ['reanalysis'],
  'data_format': 'netcdf',
  'download_format': 'unarchived',
  'variable': ['sea_surface_temperature'],
  'time': ['00:00', '06:00', '12:00', '18:00'],
  'year': ['2018'],
  'month': ['01'],
  'day': ['02', '03', '05', '06', '07', '08'],
  'area': [44.25, -38.75, 31.0, -22.5]}]

### Download

Optional: `match()` downloads what is missing.

In [4]:
my_fm.download()

### Cache

One folder per request, listed in `register.json` with its state: `pending`, `available` or `error`.

In [5]:
my_fm.show_register()

,state,variables,area,year,month,day,time,dataset
request,,,,,,,,
512d7f8053854694a9974d319946d8ca,available,[sea_surface_temperature],"[44.25, -38.75, 31.0, -22.5]",[2018],[01],"[02, 03, 05, 06, 07, 08]","[00:00, 06:00, 12:00, 18:00]",reanalysis-era5-single-levels


In [6]:
# same request: not downloaded again
my_fm.build_request()
my_fm.download()
my_fm.show_register()

,state,variables,area,year,month,day,time,dataset
request,,,,,,,,
512d7f8053854694a9974d319946d8ca,available,[sea_surface_temperature],"[44.25, -38.75, 31.0, -22.5]",[2018],[01],"[02, 03, 05, 06, 07, 08]","[00:00, 06:00, 12:00, 18:00]",reanalysis-era5-single-levels


In [7]:
# included in a request already downloaded (fewer hours): not downloaded again
my_fm.set_product(type="era5-hourly", source="remote-cds", path="./cache", selected_variables=["sst"],
                  client=client, time=["00:00", "12:00"])
my_fm.build_request()
my_fm.download()
my_fm.show_register()

,state,variables,area,year,month,day,time,dataset
request,,,,,,,,
512d7f8053854694a9974d319946d8ca,available,[sea_surface_temperature],"[44.25, -38.75, 31.0, -22.5]",[2018],[01],"[02, 03, 05, 06, 07, 08]","[00:00, 06:00, 12:00, 18:00]",reanalysis-era5-single-levels


In [8]:
# one more variable: downloaded
my_fm.set_product(type="era5-hourly", source="remote-cds", path="./cache", selected_variables=["sst", "t2m"],
                  client=client, time=["00:00", "12:00"])
my_fm.build_request()
my_fm.download()
my_fm.show_register()

2026-11-07 00:38:43,225 INFO Request ID is 028231a2-05d9-418e-b5b8-2d74430bb152
2026-11-07 00:38:43,310 INFO status has been updated to successful


,state,variables,area,year,month,day,time,dataset
request,,,,,,,,
512d7f8053854694a9974d319946d8ca,available,[sea_surface_temperature],"[44.25, -38.75, 31.0, -22.5]",[2018],[01],"[02, 03, 05, 06, 07, 08]","[00:00, 06:00, 12:00, 18:00]",reanalysis-era5-single-levels
d1fba09aa72e46d88017c0bc91237fb6,available,"[sea_surface_temperature, 2m_temperature]","[44.25, -38.75, 31.0, -22.5]",[2018],[01],"[02, 03, 05, 06, 07, 08]","[00:00, 12:00]",reanalysis-era5-single-levels


`clear_cache=True` in `set_product` empties the cache before downloading.

### Antimeridian

Points on both sides of 180°: the whole band of latitude is requested.

In [9]:
pacific = pd.DataFrame({
    "longitude": [179.5, -179.5],
    "latitude":  [10.0, 10.5],
    "date": pd.to_datetime(["2013-03-18 12:00", "2013-03-18 18:00"]),
})

pacific_fm = FloatMatcher()
pacific_fm.set_points(pacific)
pacific_fm.set_product(type="era5-hourly", source="remote-cds", path="./cache",
                       selected_variables=["sst"], client=client)
pacific_fm.build_request()[0]["area"]  # north, west, south, east

[11.0, -180.0, 9.5, 179.75]

### Without network

`FakeCDSClient` answers like CDS, with empty files: to try the requests and the cache without account.

In [10]:
fake = FakeCDSClient()

offline_fm = FloatMatcher()
offline_fm.set_points(df)
offline_fm.set_product(type="era5-hourly", source="remote-cds", path="./cache_offline",
                       selected_variables=["sst"], client=fake)
offline_fm.build_request()
offline_fm.download()
len(fake.sent), offline_fm.show_register()

(1,
                                       state                  variables  \
 request                                                                  
 eccd1a99b18249c28df67365c1b15455  available  [sea_surface_temperature]   
 
                                                           area    year month  \
 request                                                                        
 eccd1a99b18249c28df67365c1b15455  [44.25, -38.75, 31.0, -22.5]  [2018]  [01]   
 
                                                        day  \
 request                                                      
 eccd1a99b18249c28df67365c1b15455  [02, 03, 05, 06, 07, 08]   
 
                                                                                time  \
 request                                                                               
 eccd1a99b18249c28df67365c1b15455  [00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:...   
 
                                                         dataset  


Over the cost limit of CDS, the request is split by year, then by month (here with the fake cost of the fake client).

In [11]:
years = pd.DataFrame({
    "longitude": [-30.0, -29.0, -28.0],
    "latitude":  [40.0, 41.0, 42.0],
    "date": pd.to_datetime(["2016-03-01", "2017-07-01", "2018-11-01"]),
})

offline_fm.set_points(years)
offline_fm.set_product(type="era5-hourly", source="remote-cds", path="./cache_offline",
                       selected_variables=["sst", "t2m", "u10", "v10"], client=FakeCDSClient())
requests = offline_fm.build_request()
len(requests)

18

In [12]:
# even a single month over the limit: reduce area, variables or hours
offline_fm.set_product(type="era5-hourly", source="remote-cds", path="./cache_offline",
                       selected_variables=["sst"], client=FakeCDSClient(limit=1))
try:
    offline_fm.build_request()
except ValueError as error:
    print(error)

a single month exceeds limit, consider reduce area or variables or hours


A request refused by CDS or interrupted is set to `error` in the register, and downloaded fresh the next time.